# Text Mining & Search Project - Data Science Master's Degree 25/26
## Preprocessing, Text Representation, Classification, and Clustering on Amazon Fine Food Reviews

* Junaid Ahmed (923714)
* Summan Gul (925663)


# Introduction
[The Amazon Fine Food Reviews](https://https://www.kaggle.com/datasets/snap/amazon-fine-food-reviews) dataset represents a significant collection of unstructured data from one of the world's largest e-commerce platforms. Analyzing this data is essential for understanding consumer behavior and product reception in the food industry.

* Problem Statement

  The sheer volume of reviews makes manual inspection impossible. The main challenge is to develop an automated pipeline that can handle "noisy" text to extract sentiment and identify common themes accurately.


*  Objectives
   
   The project aims to perform two core text mining tasks:
    1.   **Classification:** Building a model to predict whether a review is positive or negative.
    2.   **Clustering:** Grouping similar reviews to discover hidden topics (eg, taste, packaging, or delivery).





*   Methodology

    We adopt a comparative approach to evaluate how different text representations affect model performance. We compare:



     *   **TF-IDF:** A frequency-based statistical representation.
     *   **Word2Vec**: A dense vector representation capturing semantic meaning.


The workflow follows a standard pipeline: Data Sampling$\rightarrow$Preprocessing$\rightarrow$Vectorization$\rightarrow$Model Training$\rightarrow$Evaluation.


   






The dataset consists of the following variables:



*   **Id** → Row Index.

*   **ProductId** → Unique identifier for the product (ASIN).

*   **User ID** → Unique identifier for the user.

*   **Profile Name** → Profile name of the user.

*   **HelpfulnessNumerator** → Number of users who found the review helpful.
*   **HelpfulnessDenominator** → Total number of users who indicated helpfulness.


*   **Score** → Rating between 1 and 5 stars.


*   **Time** → Timestamp for the review (Unix time).


*   **Summary** → A brief summary of the review.


*   **Text** → The full plaintext review.



# Installation and Library Imports
This will installs any missing tools and imports the libraries required for data handling, NLP, and machine learning.

In [2]:
# 1. Install necessary libraries (if not already present)
!pip install gensim nltk scikit-learn pandas matplotlib seaborn

# 2. Import standard libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import re

# 3. Import NLP and Machine Learning tools
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, silhouette_score
from gensim.models import Word2Vec

# 4. Download NLTK data for text processing
nltk.download(['punkt', 'stopwords', 'wordnet'])

print("All libraries installed and imported successfully.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 38.2 MB/s eta 0:00:00


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


All libraries installed and imported successfully.


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...


#Data Loading and Labeling
We mount your drive and load the Amazon dataset. We then create binary labels for classification based on the review scores.

In [5]:
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')

# Correct path based on your screenshot
file_path = '/content/drive/My Drive/TMS_Project_Junaid/data/Reviews.csv'

# Load 20,000 rows for efficient processing
df = pd.read_csv(file_path).head(20000)

# Create labels: Ignore neutral 3-star reviews
# 4-5 stars = 1 (Positive), 1-2 stars = 0 (Negative)
df = df[df['Score'] != 3]
df['Sentiment'] = df['Score'].apply(lambda x: 1 if x > 3 else 0)

# Keep only the columns needed for text mining
df = df[['Text', 'Sentiment']]
print("Data loaded and labeled successfully.")

Mounted at /content/drive
Data loaded and labeled successfully.


#Preprocessing (Cleaning)
We clean the text to remove HTML and non-letter characters, then apply lemmatization to get the root of each word.


In [6]:
nltk.download('punkt_tab')

stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def clean_text(text):
    # Remove HTML tags and convert to lowercase
    text = re.sub(r'<.*?>', '', str(text).lower())
    # Remove non-alphabetic characters
    text = re.sub(r'[^a-z\s]', '', text)
    # Split into words
    tokens = nltk.word_tokenize(text)
    # Remove stopwords and simplify words to root form
    cleaned = [lemmatizer.lemmatize(w) for w in tokens if w not in stop_words]
    return " ".join(cleaned)

# Process the reviews
df['Cleaned_Text'] = df['Text'].apply(clean_text)
print("Text cleaning complete.")

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


Text cleaning complete.


#Feature Extraction (Comparison)

We convert text into numbers using TF-IDF (word importance) and Word2Vec (word meaning) for comparison.

In [7]:
# 1. TF-IDF Representation
tfidf = TfidfVectorizer(max_features=2500)
X_tfidf = tfidf.fit_transform(df['Cleaned_Text'])

# 2. Word2Vec Representation
tokenized_text = [doc.split() for doc in df['Cleaned_Text']]
w2v_model = Word2Vec(sentences=tokenized_text, vector_size=100, window=5, min_count=2)

# Average the vectors for each review
def get_avg_vec(tokens):
    vecs = [w2v_model.wv[w] for w in tokens if w in w2v_model.wv]
    return np.mean(vecs, axis=0) if vecs else np.zeros(100)

X_w2v = np.array([get_avg_vec(t) for t in tokenized_text])
print("Representations created.")

Representations created.


#Classification & Accuracy
We train a model on both versions of the data and compare the accuracy scores.


In [8]:
y = df['Sentiment']

# Split data into Train and Test sets
X_train_t, X_test_t, y_train, y_test = train_test_split(X_tfidf, y, test_size=0.2, random_state=42)
X_train_w, X_test_w, _, _ = train_test_split(X_w2v, y, test_size=0.2, random_state=42)

# Train and predict
model_tfidf = LogisticRegression(max_iter=1000).fit(X_train_t, y_train)
model_w2v = LogisticRegression(max_iter=1000).fit(X_train_w, y_train)

print(f"TF-IDF Accuracy: {accuracy_score(y_test, model_tfidf.predict(X_test_t)):.4f}")
print(f"Word2Vec Accuracy: {accuracy_score(y_test, model_w2v.predict(X_test_w)):.4f}")

TF-IDF Accuracy: 0.8962
Word2Vec Accuracy: 0.8635


#Informative Features

This generates the word list you need for your report, matching the style of your reference image.

In [9]:
# Show which words influence sentiment the most
words = tfidf.get_feature_names_out()
coeffs = model_tfidf.coef_[0]
indices = np.argsort(coeffs)

print("--- TOP NEGATIVE WORDS ---")
for i in indices[:15]:
    print(f"{coeffs[i]:.4f}\t{words[i]}")

print("\n--- TOP POSITIVE WORDS ---")
for i in indices[-15:][::-1]:
    print(f"{coeffs[i]:.4f}\t{words[i]}")

--- TOP NEGATIVE WORDS ---
-5.0769	disappointed
-3.9870	terrible
-3.6956	horrible
-3.5660	return
-3.4952	weak
-3.4720	disappointing
-3.4296	waste
-3.4054	worst
-3.2383	money
-3.1723	ok
-3.1009	bad
-3.0963	awful
-3.0737	hoping
-2.9810	yuck
-2.9328	threw

--- TOP POSITIVE WORDS ---
6.7936	great
5.6470	love
4.8617	best
4.3042	delicious
4.0778	good
3.8639	excellent
3.7437	nice
3.6309	perfect
3.5576	favorite
3.2926	wonderful
2.8633	highly
2.7472	tasty
2.7156	snack
2.6733	find
2.6386	smooth


In [10]:
# --- CHUNK 7: K-MEANS CLUSTERING ---
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# 1. Train KMeans with 5 clusters using your TF-IDF data
num_clusters = 5
kmeans = KMeans(n_clusters=num_clusters, random_state=42, n_init=10)
kmeans.fit(X_tfidf)

# 2. Evaluate the clustering quality using Silhouette Score
score = silhouette_score(X_tfidf, kmeans.labels_, sample_size=5000, random_state=42)
print(f"Clustering Silhouette Score: {score:.4f}")

# 3. Print the top words for each cluster to see the hidden topics
print("\n--- TOP WORDS PER CLUSTER ---")
order_centroids = kmeans.cluster_centers_.argsort()[:, ::-1]
terms = tfidf.get_feature_names_out()

for i in range(num_clusters):
    print(f"Cluster {i}: ", end="")
    top_words = [terms[ind] for ind in order_centroids[i, :8]]
    print(", ".join(top_words))

Clustering Silhouette Score: 0.0125

--- TOP WORDS PER CLUSTER ---
Cluster 0: product, great, like, good, taste, love, flavor, one
Cluster 1: coffee, cup, like, flavor, good, blend, taste, strong
Cluster 2: chip, peanut, butter, flavor, bag, like, calorie, taste
Cluster 3: tea, green, grey, taste, flavor, earl, like, great
Cluster 4: dog, treat, food, love, product, like, one, newman
